In [5]:
%env SM_FRAMEWORK=tf.keras
!pip install segmentation-models

env: SM_FRAMEWORK=tf.keras


## Libraries and variables

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, train_test_split
import segmentation_models as sm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
tf.config.run_functions_eagerly(True)


Bad key text.latex.preview in file C:\Users\SOFIA\Anaconda3\lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test.mplstyle, line 123 ('text.latex.preview : False')
You probably need to get an updated matplotlibrc file from
https://github.com/matplotlib/matplotlib/blob/v3.5.3/matplotlibrc.template
or from the matplotlib source distribution

Bad key mathtext.fallback_to_cm in file C:\Users\SOFIA\Anaconda3\lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test.mplstyle, line 155 ('mathtext.fallback_to_cm : True  # When True, use symbols from the Computer Modern')
You probably need to get an updated matplotlibrc file from
https://github.com/matplotlib/matplotlib/blob/v3.5.3/matplotlibrc.template
or from the matplotlib source distribution

Bad key savefig.jpeg_quality in file C:\Users\SOFIA\Anaconda3\lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test.mplstyle, line 418 ('savefig.jpeg_quality: 95       # when a jpeg is saved, the default quality parameter.')
You proba

Segmentation Models: using `tf.keras` framework.


In [2]:
IMG_HEIGHT = 512
IMG_WIDTH = 512
IMG_CHANNEL = 3
IMG_PATH = '../Dataset/Images/'
MASK_PATH = '../Dataset/GT/'

BATCH_SIZE = 1
EPOCHS = 50
BACKBONE = 'inceptionv3'

## Data preparation

In [4]:
# Geting data
img_files = os.listdir(IMG_PATH)
mask_files = os.listdir(MASK_PATH)

X = np.zeros((len(img_files), IMG_WIDTH, IMG_HEIGHT, IMG_CHANNEL), dtype=np.float32)
Y = np.zeros((len(img_files), IMG_WIDTH, IMG_HEIGHT, 1), dtype=np.float32)


for i, img_file in enumerate(img_files):
    
    # Image
    image = cv2.imread(IMG_PATH + img_file)
    image = cv2.resize(image, (IMG_HEIGHT, IMG_WIDTH), interpolation = cv2.INTER_LINEAR)
    X[i,:,:,:] = image/255.

    # Mask
    mask = cv2.imread(MASK_PATH + 'gt_c' + str(img_file[:-4]) + '.jpg')
    mask = cv2.resize(mask, (IMG_HEIGHT, IMG_WIDTH), interpolation = cv2.INTER_LINEAR)
    Y[i,:,:,:] = np.expand_dims(mask[:,:,0], axis=-1)/255.

# Spliting dataset train/test
idx_train, idx_test = train_test_split(range(0, len(img_files)), test_size = 0.3, random_state = 42)

kf = KFold(n_splits = 10)
i = 1

import pickle
for train_index, val_index in kf.split(X):
    with open('../Dataset/' + str(i) + '_split.pickle' , 'wb') as f:
        pickle.dump((train_index, val_index), f)
        
    i += 1

In [5]:
with open('../Dataset/idx_test.pickle' , 'wb') as f:
    pickle.dump(idx_test, f)